# 3. Evaluate (GPU notebook)

- Settings: GPU (one is enough), Internet **On** (to clone the published baselines).
- Inputs:
  - the cache dataset (with the test split);
  - the training notebook's output (for its runs);
  - the code dataset;
  - for the published baselines, the UnCRtainTS checkpoint folder and GLF-CR's `CR_net.pth`,
    uploaded as a private dataset. See `baselines/external.md` for which UnCRtainTS folder.
- Results go to `/kaggle/working/eval`. Save the version to keep them.

In [ ]:
import os, glob, shutil, time
T0 = time.time()                      # session start, used to stop training before Kaggle's 12 h limit
REPO_URL = ''                         # optional: git URL of the DieWolke repo, if it is pushed somewhere
CODE = '/kaggle/working/DieWolke'
if not os.path.exists(CODE):
    if REPO_URL:
        !git clone -q {REPO_URL} {CODE}
    else:  # the code uploaded as a Kaggle dataset (the DieWolke folder, without data or runs)
        # prefer the code dataset over stray copies (e.g. an old DieWolke folder inside a cache dataset)
        hits = sorted(glob.glob('/kaggle/input/**/diewolke/__init__.py', recursive=True),
                      key=lambda p: ('diewolke-code' not in p, len(p)))
        assert hits, 'Add the DieWolke code as an input dataset (Add Input > Your Datasets), or set REPO_URL'
        shutil.copytree(os.path.dirname(os.path.dirname(hits[0])), CODE,
                        ignore=shutil.ignore_patterns('*.npy', 'runs', '.git', '.venv', '__pycache__'))
        print('code copied from', os.path.dirname(os.path.dirname(hits[0])))
%cd {CODE}
!grep -q index_floor diewolke/losses.py && echo 'code: NDVI fix present' || echo 'code: OLD (no NDVI fix)'
!python kaggle/kaggle_tools.py env

In [ ]:
!python kaggle/kaggle_tools.py restore
CACHE = !python kaggle/kaggle_tools.py find-cache --split test --copy_to /kaggle/temp/cache
CACHE = CACHE[-1]; print('test cache:', CACHE)
RUNS, EVAL = '/kaggle/working/runs', '/kaggle/working/eval'
os.makedirs(EVAL, exist_ok=True)
names = [n for n in ['full', 'matched_baseline'] if os.path.exists(f'{RUNS}/{n}/best.pt')]
abl = [n for n in ['full_20k', 'no_bga', 'no_ndvi', 'no_unc'] if os.path.exists(f'{RUNS}/{n}/best.pt')]
VAL = '--calibrate_on val' if os.path.exists(f'{CACHE}/val_clear.npy') else '--calibrate_on ""'
print(names, abl)

In [ ]:
MAIN = ' '.join(f'{RUNS}/{n}' for n in names); ABL = ' '.join(f'{RUNS}/{n}' for n in abl)
!python -m diewolke.evaluate --runs {MAIN} --cache {CACHE} --split test --mode real {VAL} --save_predictions --out {EVAL}/test_real
!python -m diewolke.evaluate --runs {MAIN} --cache {CACHE} --split test --mode synthetic {VAL} --out {EVAL}/test_synth
!python -m diewolke.evaluate --runs {ABL} --cache {CACHE} --split test --mode real {VAL} --out {EVAL}/ablations_real
# C5 two-stage: the full model with the Stage 1 masker's masks instead of s2cloudless
!python -m diewolke.evaluate --runs {RUNS}/full --masker {RUNS}/masker --mask_source masker --cache {CACHE} --split test --mode synthetic {VAL} --out {EVAL}/two_stage_synth
print(open(f'{EVAL}/test_real/report.md').read())

**Published baselines and XGBoost.** Set the two weight paths to where your uploaded weights appear under `/kaggle/input`.

In [ ]:
UNCR_WEIGHTS = ''   # folder that contains <experiment>/conf.json and model.pth.tar
UNCR_EXP = ''       # a checkpoint folder whose conf.json has "pretrain": true
GLF_CKPT = ''       # .../CR_net.pth
!pip install -q fvcore timm
preds = []
if UNCR_WEIGHTS and UNCR_EXP:
    !git clone -q https://github.com/PatrickTUM/UnCRtainTS /kaggle/temp/UnCRtainTS
    !python baselines/run_external.py uncrtaints --repo /kaggle/temp/UnCRtainTS --weights {UNCR_WEIGHTS} --experiment {UNCR_EXP} --cache {CACHE} --split test --mode real --out {EVAL}/pred_uncrtaints.npy
    preds.append(f'uncrtaints={EVAL}/pred_uncrtaints.npy,{EVAL}/scale_uncrtaints.npy')
if GLF_CKPT:
    !git clone -q https://github.com/xufangchn/GLF-CR /kaggle/temp/GLF-CR
    !python baselines/run_external.py glfcr --repo /kaggle/temp/GLF-CR --ckpt {GLF_CKPT} --cache {CACHE} --split test --mode real --out {EVAL}/pred_glfcr.npy
    preds.append(f'glfcr={EVAL}/pred_glfcr.npy')
TRAIN_CACHE = !python kaggle/kaggle_tools.py find-cache --split train --copy_to /kaggle/temp/cache
!python baselines/xgb_reconstruct.py --cache {CACHE} --split test --train_cache {TRAIN_CACHE[-1]} --mode real --out {EVAL}/pred_xgb.npy
preds.append(f'xgb={EVAL}/pred_xgb.npy')
PREDS = ' '.join(preds)
!python -m diewolke.evaluate --runs {MAIN} --preds {PREDS} --cache {CACHE} --split test --mode real {VAL} --out {EVAL}/test_real_baselines
print(open(f'{EVAL}/test_real_baselines/report.md').read())

In [ ]:
# keep the saved output small: drop the large prediction arrays if space is tight (reports and CSVs stay)
!du -sh {EVAL}
# !rm -f {EVAL}/*.npy {EVAL}/*/pred_*.npy {EVAL}/*/scale_*.npy